# Experiment 12: Phase / Amplitude / Noise Robustness

Progressively increase variability and corruption: random phase, random amplitude, then additive noise.

In [ ]:
from pathlib import Path
import sys


def find_repo_root(start: Path | None = None) -> Path:
    current = Path.cwd().resolve() if start is None else Path(start).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "src" / "config.py").exists() and (candidate / "README.md").exists():
            return candidate
    raise RuntimeError("Could not locate repository root.")


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(f"Repo root: {REPO_ROOT}")


In [ ]:
import numpy as np
import pandas as pd

try:
    from tqdm.auto import tqdm
except Exception:
    def tqdm(iterable=None, *args, **kwargs):
        return iterable if iterable is not None else []

from src import ExperimentConfig, run_experiment


def overall_row(results: dict) -> pd.Series:
    summary_df = results["summary_df"]
    return summary_df.loc[summary_df["set_idx"] == "overall"].iloc[0]


def collect_overall_rows(result_map: dict[str, dict], columns: list[str] | None = None) -> pd.DataFrame:
    rows = []
    for run_label, result in result_map.items():
        row = overall_row(result).copy()
        row["run_label"] = run_label
        rows.append(row)
    df = pd.DataFrame(rows)
    if columns is not None:
        return df[["run_label", *columns]]
    return df


In [ ]:
BASE_CONFIG = dict(
    time_mode="discrete",
    SEQ_LEN=4096,
    theta_min=0.05 * np.pi,
    theta_max=0.85 * np.pi,
    MIN_DELTA_THETA=0.12 * np.pi,
    HIDDEN_DIM=128,
    BOTTLENECK_MULTIPLIER=4,
    NUM_EXPERIMENTS=20,
    SEEDS_PER_FREQ=5,
    VAL_RATIO=0.2,
    TEST_RATIO=0.2,
    NORMALIZE_H_COLUMNS=False,
    VERBOSE=False,
    MAKE_PLOTS=False,
)


In [ ]:
CONDITIONS = {
    "phase_only": dict(RANDOM_PHASE=True, RANDOM_AMPLITUDE=False, USE_NOISE=False),
    "phase_amp": dict(RANDOM_PHASE=True, RANDOM_AMPLITUDE=True, USE_NOISE=False),
    "noise_30db": dict(RANDOM_PHASE=True, RANDOM_AMPLITUDE=True, USE_NOISE=True, SNR_DB=30.0),
    "noise_20db": dict(RANDOM_PHASE=True, RANDOM_AMPLITUDE=True, USE_NOISE=True, SNR_DB=20.0),
    "noise_10db": dict(RANDOM_PHASE=True, RANDOM_AMPLITUDE=True, USE_NOISE=True, SNR_DB=10.0),
    "noise_5db": dict(RANDOM_PHASE=True, RANDOM_AMPLITUDE=True, USE_NOISE=True, SNR_DB=5.0),
}


In [ ]:
results = {}
for condition_name, condition_kwargs in tqdm(list(CONDITIONS.items()), desc="experiment 12 conditions"):
    cfg = ExperimentConfig(
        **BASE_CONFIG,
        NUM_FREQS=4,
        LAG=32,
        MODEL_ID="AN002_NO_BN_TANH",
        LR=1e-3,
        EPOCHS=1500,
        **condition_kwargs,
    )
    results[condition_name] = run_experiment(cfg)

collect_overall_rows(
    results,
    columns=[
        "test_mse_mean",
        "test_align_coverage_full_mean",
        "test_recon_r2_qf_from_h_mean",
        "train_rank_threshold_mean",
        "test_rank_threshold_mean",
        "train_rank_entropy_mean",
        "test_rank_entropy_mean",
        "train_h_numerical_dim_mean",
        "test_h_numerical_dim_mean",
        "test_input_snr_db_mean",
        "test_output_snr_db_mean",
        "test_snr_gain_db_mean",
    ],
)
